
# Notebook 03b — Metric Views: the governed semantic layer

**The single most important idea in this workshop:** a Genie agent is only as good as the data foundation under it. **Move the logic to the left** — define your KPIs and joins **once**, in a governed object in Unity Catalog, instead of hoping Genie re-derives them correctly on every question.

A **metric view** is that object. It pins:
- the **join paths** between fact and dimension tables,
- the **exact formula** for each KPI (OEE, first-pass yield, scrap rate, …),
- the **dimensions** you're allowed to slice by.

Genie (and dashboards, and SQL users) then all read the *same* trusted definitions.

| | Tables (03, 04) | Metric view (here) |
|---|---|---|
| KPI formula | Genie must learn it (examples/measures) | **Fixed** in the view |
| Joins | Genie must pick them | **Fixed** in the view |
| Governance | per-table grants | one governed object, reusable |
| Flexibility | open-ended exploration | limited to the view's perimeter |

**Best for** business-critical, must-be-right KPIs. Pair it with tables when users also need open-ended exploration. This notebook builds one metric view and a Genie agent on top of it (**Agent C** — compared in notebook 08).

> **Verify the DDL in your workspace.** Metric-view YAML syntax has evolved across releases; if `CREATE VIEW ... WITH METRICS` errors, check the current syntax in the Databricks docs for your workspace version.

**Before you start:** run notebooks **02** (data) and **03** (baseline agent).

**Compute:** Serverless.

In [ ]:
%run ./00_workshop_config

In [ ]:
from databricks.sdk import WorkspaceClient
import re
import json
import uuid
import requests

w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(space_id: str) -> str:
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"


warehouse_id = None
for wh in w.warehouses.list():
    state = str(wh.state).upper() if wh.state else ""
    if state in ("RUNNING", "STARTING"):
        warehouse_id = wh.id
        break
if not warehouse_id:
    whs = list(w.warehouses.list())
    warehouse_id = whs[0].id if whs else None
if not warehouse_id:
    raise RuntimeError("No SQL warehouse found. Create or start one, then re-run.")
print("Warehouse:", warehouse_id)

## 1. Create the metric view

`mv_line_quality` sits on top of `quality_metrics_daily` (the daily aggregate) and joins to `production_lines` and `plants`. It defines the KPI **measures** and the **dimensions** you may group by. Notice the formulas — scrap rate and defect rate are pinned here, so nobody has to re-derive them.

In [ ]:
ddl = f"""
CREATE OR REPLACE VIEW {METRIC_VIEW_LINE_QUALITY}
WITH METRICS
LANGUAGE YAML
AS $$
version: 0.1
source: {fqn}.quality_metrics_daily
joins:
  - name: line
    source: {fqn}.production_lines
    on: source.production_line_id = line.line_id
  - name: plant
    source: {fqn}.plants
    on: source.plant_id = plant.plant_id
dimensions:
  - name: date
    expr: date
  - name: plant_name
    expr: plant.plant_name
  - name: state
    expr: plant.state
  - name: line_name
    expr: line.line_name
  - name: product_type
    expr: line.product_type
measures:
  - name: avg_oee
    expr: AVG(oee_score)
  - name: avg_first_pass_yield
    expr: AVG(first_pass_yield)
  - name: total_units_produced
    expr: SUM(units_produced)
  - name: total_defects
    expr: SUM(defects_found)
  - name: scrap_rate_pct
    expr: 100.0 * SUM(scrap_count) / NULLIF(SUM(units_produced), 0)
  - name: defect_rate_pct
    expr: 100.0 * SUM(defects_found) / NULLIF(SUM(units_produced), 0)
  - name: total_downtime_minutes
    expr: SUM(downtime_minutes)
$$
"""
spark.sql(ddl)
print("Created metric view:", METRIC_VIEW_LINE_QUALITY)

## 2. Query it with `MEASURE()`

You query a metric view by selecting dimensions and wrapping measures in `MEASURE(...)`. The engine applies the pinned joins and formulas — the same numbers every time, for everyone.

In [ ]:
display(spark.sql(f"""
SELECT
  plant_name,
  ROUND(MEASURE(avg_oee), 3)            AS avg_oee,
  ROUND(MEASURE(scrap_rate_pct), 2)     AS scrap_rate_pct,
  ROUND(MEASURE(defect_rate_pct), 2)    AS defect_rate_pct,
  CAST(MEASURE(total_units_produced) AS BIGINT) AS units_produced
FROM {METRIC_VIEW_LINE_QUALITY}
GROUP BY plant_name
ORDER BY avg_oee DESC
"""))

## 3. Build a Genie agent on the metric view

**Agent C.** Its only data source is the metric view — so the joins and KPI math are guaranteed correct by construction. Genie's job shrinks to "pick dimensions and measures," which is exactly where it's most reliable.

In [ ]:
def build_serialized_metric_view():
    mv_instr = (
        "Answer questions using the governed metric view only. "
        "Select dimensions and wrap metrics in MEASURE(<measure_name>), grouping by "
        "the requested dimensions. The metric view already encodes the correct joins "
        "and KPI formulas — do not recompute them from base tables."
    )
    return json.dumps({
        "version": 2,
        "config": {"sample_questions": []},
        "data_sources": {
            "tables": [],
            "metric_views": [{
                "identifier": METRIC_VIEW_LINE_QUALITY,
                "description": [
                    "Governed line-level quality metrics: OEE, first-pass yield, "
                    "scrap and defect rates, downtime — by plant, state, line, product, and date."
                ],
            }],
        },
        "instructions": {
            "text_instructions": [{"id": uuid.uuid4().hex, "content": [mv_instr]}],
            "example_question_sqls": [],
        },
    })


def _list_spaces():
    r = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
    r.raise_for_status()
    return r.json().get("spaces", [])


def create_or_update_genie_space(title, description, serialized_space_str):
    for s in _list_spaces():
        if s.get("title") == title:
            sid = s.get("space_id") or s.get("id")
            requests.patch(
                f"{host}/api/2.0/genie/spaces/{sid}", headers=headers,
                json={"title": title, "description": description,
                      "warehouse_id": warehouse_id, "serialized_space": serialized_space_str},
            )
            print(f"Updated existing: {title!r} -> {sid}")
            return sid, genie_ui_room_url(sid)
    resp = requests.post(
        f"{host}/api/2.0/genie/spaces", headers=headers,
        json={"title": title, "description": description,
              "warehouse_id": warehouse_id, "serialized_space": serialized_space_str},
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError(f"Genie create failed {resp.status_code}: {resp.text[:800]}")
    sid = resp.json().get("space_id") or resp.json().get("id")
    print(f"Created: {title!r} -> {sid}")
    return sid, genie_ui_room_url(sid)


mv_id, mv_url = create_or_update_genie_space(
    GENIE_TITLE_METRIC_VIEW, GENIE_DESC_METRIC_VIEW, build_serialized_metric_view()
)

save_config_keys([
    {"key": CFG_KEY_METRIC_VIEW, "value": mv_id,
     "space_name": GENIE_TITLE_METRIC_VIEW, "space_url": mv_url},
])

print()
print("Metric-view agent:", mv_url)
print('Try: "What is the average OEE and scrap rate by plant?" — the metric view')
print("guarantees the joins and formulas, so the answer is deterministic.")

## Next

- **04 — Knowledge Store:** curate the *tables* with measures, filters, fields, joins, synonyms, and example SQL (the primary agent).
- **08** compares Baseline vs. **Metric View** vs. Knowledge Store.

**Takeaway for your own domain:** for the KPIs your business runs on, define them once in a metric view. It's the highest-leverage thing you can do before pointing Genie at your data.